In [90]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import tree
import joblib
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, classification_report

In [91]:
d = pd.read_csv(r"odor_dataset_final.csv")
d

,Ethanol,Isopropyl Alcohol,Methanol,Butane,Propane,Isobutane,Dimethyl Ether,Formaldehyde,Benzaldehyde,Cinnamaldehyde,Acetone,Musk Ketone,Ethyl Acetate,Amyl Acetate,Toluene,Xylene,Class
0,23.59,10.98,17.04,15.69,9.02,10.23,1.04,13.55,4.70,9.03,5.87,9.94,10.11,2.44,7.16,11.61,Alcoholic
1,11.12,1.94,13.53,20.38,10.44,9.31,5.52,10.47,7.52,16.80,7.83,7.72,21.33,18.10,2.38,9.55,Fruity
2,10.24,3.76,12.50,16.63,9.45,6.75,2.57,12.07,8.51,19.49,10.95,11.47,17.23,21.84,4.89,10.11,Fruity
3,9.12,9.09,1.78,16.65,11.06,6.73,3.47,11.72,1.49,7.70,6.70,9.98,10.33,8.21,7.31,6.25,Gasoline-like
4,6.27,2.42,13.33,15.74,6.65,1.55,2.09,12.17,2.53,11.64,15.02,7.02,6.25,4.80,9.22,14.48,Chemical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,18.31,5.44,18.10,3.38,9.00,0.23,0.85,12.92,8.08,9.19,9.66,11.66,11.78,12.26,2.05,11.30,Sweet
4996,16.46,11.93,17.78,2.07,8.88,7.77,0.10,5.48,7.12,18.15,3.22,7.02,17.35,22.36,5.01,8.15,Fruity
4997,17.12,10.28,23.18,4.73,10.14,6.87,9.65,12.02,5.71,8.45,6.49,10.71,14.43,9.56,4.87,7.77,Alcoholic
4998,13.73,9.67,14.07,4.20,9.16,8.49,7.81,8.51,5.54,14.78,6.74,5.12,16.71,18.29,7.57,7.33,Fruity


In [92]:
X = d.drop(columns=["Class"])  #
y = d["Class"]  

In [93]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [94]:
joblib.dump(label_encoder, "label_encoder.pkl")

['label_encoder.pkl']

In [95]:
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'Alcoholic': np.int64(0), 'Chemical': np.int64(1), 'Fruity': np.int64(2), 'Gasoline-like': np.int64(3), 'Sweet': np.int64(4)}


In [96]:

X_train, X_test, y_train, ytest = train_test_split(X, y_encoded, test_size=0.1, random_state=100)

In [97]:
model = DecisionTreeClassifier(criterion="entropy", max_depth=5)

In [98]:
model.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=5)

In [99]:
tree_rules = export_text(model, feature_names=X.columns.to_list())
print("\nDecision Tree Rules:\n")
print(tree_rules)


Decision Tree Rules:

|--- Ethyl Acetate <= 13.28
|   |--- Formaldehyde <= 10.88
|   |   |--- Isopropyl Alcohol <= 9.24
|   |   |   |--- Propane <= 9.32
|   |   |   |   |--- Musk Ketone <= 6.64
|   |   |   |   |   |--- class: 3
|   |   |   |   |--- Musk Ketone >  6.64
|   |   |   |   |   |--- class: 4
|   |   |   |--- Propane >  9.32
|   |   |   |   |--- Butane <= 11.14
|   |   |   |   |   |--- class: 4
|   |   |   |   |--- Butane >  11.14
|   |   |   |   |   |--- class: 3
|   |   |--- Isopropyl Alcohol >  9.24
|   |   |   |--- Ethanol <= 16.09
|   |   |   |   |--- Butane <= 12.60
|   |   |   |   |   |--- class: 4
|   |   |   |   |--- Butane >  12.60
|   |   |   |   |   |--- class: 3
|   |   |   |--- Ethanol >  16.09
|   |   |   |   |--- Methanol <= 13.91
|   |   |   |   |   |--- class: 4
|   |   |   |   |--- Methanol >  13.91
|   |   |   |   |   |--- class: 0
|   |--- Formaldehyde >  10.88
|   |   |--- Toluene <= 7.55
|   |   |   |--- Butane <= 11.51
|   |   |   |   |--- Acetone <= 1

In [100]:
joblib.dump(model, "decision_tree_perfume_sir.pkl")
print("Model saved successfully!")

Model saved successfully!


In [101]:
print(d["Class"].value_counts())  

Class
Alcoholic        1000
Fruity           1000
Gasoline-like    1000
Chemical         1000
Sweet            1000
Name: count, dtype: int64


In [102]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(ytest, y_pred))

Accuracy: 0.756


In [103]:
f1 = f1_score(ytest, y_pred, average='weighted')
print("F1 Score:", f1)

F1 Score: 0.7578314796857838


In [104]:
cm = confusion_matrix(ytest, y_pred)
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_))

Confusion Matrix:
               Alcoholic  Chemical  Fruity  Gasoline-like  Sweet
Alcoholic             68         5       4             11     14
Chemical               5        83       7              3      4
Fruity                 4         0      75             10      9
Gasoline-like          9         3       2             78     11
Sweet                  6         0       4             11     74


In [105]:
# Calculate accuracy
accuracy = accuracy_score(ytest, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Print classification report
print("Classification Report:")
print(classification_report(ytest, y_pred))

Model Accuracy: 0.76
Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.67      0.70       102
           1       0.91      0.81      0.86       102
           2       0.82      0.77      0.79        98
           3       0.69      0.76      0.72       103
           4       0.66      0.78      0.71        95

    accuracy                           0.76       500
   macro avg       0.76      0.76      0.76       500
weighted avg       0.76      0.76      0.76       500

